# MVP — Previsão de Qualidade de Vinho Tinto

**Disciplina:** Machine Learning & Analytics
**Tipo de problema:** Regressão Supervisionada
**Dataset:** Wine Quality – Red Wine (UCI Machine Learning Repository)
**Repositório GitHub:** https://github.com/Eddy8080/MVP-individual-de-Machine-Learning-Analytics

---

Este notebook implementa um MVP completo de Machine Learning, seguindo o fluxo:
**Definição do problema → Exploração dos dados → Preparação → Modelagem → Avaliação → Conclusão.**

Todas as células devem ser executadas **em ordem, do início ao fim**, sem configuração adicional.


## 0. Configuração do Ambiente

Nesta célula são importadas todas as bibliotecas necessárias e definida a **semente aleatória global**
(`SEED = 42`), garantindo que os resultados sejam **reproduzíveis** em qualquer execução.


In [ ]:
# ── Manipulação e análise de dados ────────────────────────────────────────────
import pandas as pd          # leitura de CSV, DataFrames e operações tabulares
import numpy as np           # cálculos numéricos, arrays e definição de seed
import time                  # medição do tempo de treinamento dos modelos

# ── Visualização ───────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt   # criação de gráficos (histogramas, scatter, barras)
import seaborn as sns             # gráficos estatísticos avançados (heatmap, boxplot)

# ── Divisão e validação dos dados ─────────────────────────────────────────────
from sklearn.model_selection import train_test_split   # divisão treino/teste
from sklearn.model_selection import GridSearchCV       # busca de hiperparâmetros com CV

# ── Pré-processamento ──────────────────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler       # padronização (média 0, desvio 1)
from sklearn.pipeline import Pipeline                  # encadeamento de etapas de ML

# ── Modelos de Machine Learning ───────────────────────────────────────────────
from sklearn.dummy import DummyRegressor               # baseline ingênuo
from sklearn.linear_model import LinearRegression      # regressão linear clássica
from sklearn.tree import DecisionTreeRegressor         # árvore de decisão
from sklearn.ensemble import RandomForestRegressor     # ensemble de árvores (bagging)

# ── Métricas de avaliação ─────────────────────────────────────────────────────
from sklearn.metrics import mean_absolute_error        # MAE: erro médio absoluto
from sklearn.metrics import mean_squared_error         # MSE: usado para calcular RMSE
from sklearn.metrics import r2_score                   # R²: coeficiente de determinação

import warnings
warnings.filterwarnings('ignore')   # silencia avisos não críticos

# Semente global — garante os mesmos resultados em qualquer execução
SEED = 42
np.random.seed(SEED)

print('Ambiente configurado com sucesso!')
print(f'  pandas  : {pd.__version__}')
print(f'  numpy   : {np.__version__}')
print(f'  seaborn : {sns.__version__}')


## 1. Apresentação do Problema

### 1.1 Contexto

A indústria vitivinícola depende fortemente da **avaliação de qualidade** dos vinhos para
definir preços, controlar o processo produtivo e posicionar produtos no mercado.
Essa avaliação é feita por **especialistas** (somméliers), o que é caro, subjetivo e difícil de escalar.

### 1.2 Objetivo do modelo

Construir um modelo de **regressão supervisionada** que estime a pontuação de qualidade (`quality`)
de um vinho tinto com base em suas **11 propriedades físico-químicas** mensuráveis em laboratório.

### 1.3 Variável a ser prevista

- **`quality`** — pontuação inteira atribuída por somméliers (escala 0–10)
- Tratada como variável **contínua** → **problema de regressão**

### 1.4 Por que é um problema de Machine Learning?

- A relação entre propriedades físico-químicas e qualidade percebida é **complexa e não-linear**
- Há **múltiplas variáveis** interagindo simultaneamente (acidez, álcool, sulfatos, etc.)
- Um modelo treinado em dados históricos pode **aprender esses padrões** e generalizar para novos vinhos
- Permite **escalar e objetivar** a avaliação sem depender de especialistas humanos

### 1.5 Premissas, hipóteses e restrições

| # | Hipótese | Esperado |
|---|---|---|
| H1 | Maior teor alcoólico → maior qualidade | Correlação positiva |
| H2 | Maior acidez volátil → menor qualidade (sabor avinagrado) | Correlação negativa |
| H3 | Maior concentração de sulfatos → maior qualidade | Correlação positiva |
| H4 | Relações não-lineares justificam modelos de ensemble | Random Forest > Regressão Linear |

**Restrições:**
- Dados restritos a vinhos tintos da região do Minho (Portugal)
- Pontuação é subjetiva (média de 3 avaliadores)
- Não há informações sobre safra, variedade de uva, produtor ou preço


## 2. Apresentação dos Dados

### 2.1 Fonte

| Item | Detalhe |
|---|---|
| **Nome** | Wine Quality – Red Wine |
| **Repositório** | UCI Machine Learning Repository |
| **Referência** | Cortez et al., 2009 — *Modeling wine preferences by data mining* |
| **Carregamento** | URL pública do repositório GitHub do projeto (sem login, token ou upload) |

### 2.2 Atributos do dataset

| Atributo | Tipo | Descrição |
|---|---|---|
| `fixed_acidity` | Numérico | Acidez fixa (g/dm³) — ácidos tartárico, málico, cítrico |
| `volatile_acidity` | Numérico | Acidez volátil (g/dm³) — altos valores = sabor avinagrado |
| `citric_acid` | Numérico | Ácido cítrico (g/dm³) — contribui para frescor |
| `residual_sugar` | Numérico | Açúcar residual após fermentação (g/dm³) |
| `chlorides` | Numérico | Cloretos (g/dm³) — concentração de sal |
| `free_sulfur_dioxide` | Numérico | SO₂ livre (mg/dm³) — antimicrobiano e antioxidante |
| `total_sulfur_dioxide` | Numérico | SO₂ total (mg/dm³) — livre + ligado |
| `density` | Numérico | Densidade (g/cm³) — relacionada ao teor alcoólico e açúcar |
| `pH` | Numérico | Nível de acidez geral (escala 0–14) |
| `sulphates` | Numérico | Sulfatos (g/dm³) — contribuem para SO₂ |
| `alcohol` | Numérico | Teor alcoólico (% vol.) |
| **`quality`** | **Numérico** | **Variável-alvo:** pontuação do vinho (0–10) |

### 2.3 Critérios de escolha

- Dataset do **UCI Machine Learning Repository** (repositório sugerido na especificação)
- Disponível via **URL pública direta** — sem login, token ou configuração adicional
- Problema de **negócio real** (precificação e controle de qualidade industrial)
- Todas as features são **numéricas** — permite demonstrar o pipeline de pré-processamento
- **Não utilizado nas aulas da sprint** — dataset de domínio industrial/enológico específico

### 2.4 Limitações

- Dataset pequeno (~1.600 registros)
- Notas de qualidade são subjetivas e médias de poucos avaliadores
- Dados restritos a vinhos tintos da região do Minho, Portugal
- Ausência de safra, variedade de uva, produtor ou informações de mercado


In [ ]:
# URL pública do dataset no repositório GitHub do projeto
# O arquivo winequality-red.csv está na raiz do repositório
URL = (
    'https://raw.githubusercontent.com/Eddy8080/'
    'MVP-individual-de-Machine-Learning-Analytics/main/winequality-red.csv'
)

# Carregar o CSV
# IMPORTANTE: o dataset UCI Wine Quality usa ponto-e-vírgula (;) como separador
df = pd.read_csv(URL, sep=';')

# Renomear colunas: substituir espaços por underscores (convenção Python/pandas)
# Ex: 'fixed acidity' -> 'fixed_acidity'
df.columns = [col.replace(' ', '_') for col in df.columns]

print('Dataset carregado com sucesso!')
print(f'  Registros  : {df.shape[0]}')
print(f'  Atributos  : {df.shape[1]}')
print(f'  Colunas    : {list(df.columns)}')


In [ ]:
# Exibir as primeiras 10 linhas do dataset para inspeção visual inicial
# Permite verificar:
#   - se o CSV foi lido corretamente (separador ; funcionou)
#   - se os nomes das colunas estão corretos (underscores no lugar de espaços)
#   - se os valores parecem razoáveis para um dataset de vinhos
#   - se não há lixo (cabeçalho duplicado, linhas vazias, caracteres estranhos)
df.head(10)


## 3. Análise Exploratória Inicial (EDA)

Antes de modelar, precisamos compreender os dados em profundidade:
tipos de variáveis, distribuições, valores ausentes e relações com a variável-alvo.


### 3.1 Tipos de variáveis e estatísticas descritivas

In [ ]:
# Verificar o tipo de dado de cada coluna
print('=== Tipos de variáveis ===')
print(df.dtypes)
print(f'\nTotal de features numéricas: {df.select_dtypes(include=np.number).shape[1]}')


In [ ]:
# Estatísticas descritivas: média, desvio padrão, mínimo, quartis e máximo
# Permite identificar escalas, amplitude e possíveis outliers
print('=== Estatísticas descritivas ===')
df.describe().round(3)


### 3.2 Verificação de valores ausentes

In [ ]:
# Contar valores nulos por coluna — passo obrigatório antes de qualquer transformação
ausentes = df.isnull().sum()

print('=== Valores ausentes por coluna ===')
print(ausentes)
print(f'\nTotal de valores ausentes no dataset: {ausentes.sum()}')

# Decisão documentada:
if ausentes.sum() == 0:
    print('Nenhum valor ausente encontrado.')
    print('Decisão: nenhuma ação de imputação necessária.')
else:
    print('AVISO: valores ausentes encontrados — aplicar estratégia de imputação.')


### 3.3 Distribuição da variável-alvo (`quality`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Distribuição da variável-alvo: quality', fontsize=14, fontweight='bold')

# ── Gráfico 1: frequência de cada pontuação ─────────────────────────────────
# Mostra quantos vinhos receberam cada nota — permite identificar desbalanceamento
contagem = df['quality'].value_counts().sort_index()   # ordenar por nota crescente

axes[0].bar(
    contagem.index,    # eixo x: notas (3 a 8)
    contagem.values,   # eixo y: quantidade de vinhos com aquela nota
    color='steelblue',
    edgecolor='white',
    width=0.6
)
axes[0].set_title('Frequência por pontuação de qualidade')
axes[0].set_xlabel('Pontuação de qualidade')
axes[0].set_ylabel('Número de vinhos')
axes[0].set_xticks(contagem.index)

# Adicionar o número exato em cima de cada barra
for nota, qtd in zip(contagem.index, contagem.values):
    axes[0].text(nota, qtd + 5, str(qtd), ha='center', fontsize=10, fontweight='bold')

# ── Gráfico 2: boxplot do álcool por nota ────────────────────────────────────
# Testa visualmente a hipótese H1: maior álcool -> maior qualidade
df.boxplot(column='alcohol', by='quality', ax=axes[1])
axes[1].set_title('Teor alcoólico (%) por pontuação')
axes[1].set_xlabel('Pontuação de qualidade')
axes[1].set_ylabel('Álcool (% vol.)')
plt.suptitle('')   # remove o título automático gerado pelo pandas.boxplot

plt.tight_layout()
plt.show()

# Estatísticas resumidas da variável-alvo
print(f'Pontuação mínima  : {df["quality"].min()}')
print(f'Pontuação máxima  : {df["quality"].max()}')
print(f'Média             : {df["quality"].mean():.3f}')
print(f'Mediana           : {df["quality"].median():.1f}')
print(f'Desvio padrão     : {df["quality"].std():.3f}')


**Observação sobre distribuição:**
A qualidade concentra-se nas notas **5 e 6**, com poucos vinhos de qualidade muito baixa (3, 4)
ou muito alta (8). Como é um problema de regressão, não aplicamos técnicas de balanceamento.


### 3.4 Relações entre features e variável-alvo

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Relação das principais features com quality', fontsize=14, fontweight='bold')

# Features selecionadas para análise visual
features_analisar = [
    'alcohol', 'volatile_acidity', 'sulphates',
    'citric_acid', 'total_sulfur_dioxide', 'pH'
]

for ax, feature in zip(axes.flatten(), features_analisar):
    # Dispersão: feature no eixo x, quality no eixo y
    ax.scatter(df[feature], df['quality'], alpha=0.25, color='steelblue', s=18)
    ax.set_xlabel(feature)
    ax.set_ylabel('quality')
    ax.set_title(f'{feature} vs quality')

    # Linha de tendência linear (reta de ajuste) para facilitar a leitura visual
    coef = np.polyfit(df[feature], df['quality'], 1)   # calcula coeficientes da reta
    reta = np.poly1d(coef)                              # cria função polinomial de grau 1
    x_ord = np.sort(df[feature].values)                # ordenar para plotar a reta
    ax.plot(x_ord, reta(x_ord), 'r--', alpha=0.8, linewidth=1.5)

plt.tight_layout()
plt.show()


### 3.5 Matriz de correlação

In [ ]:
# Correlação de Pearson: mede a relação linear entre todas as variáveis numéricas
# Valores próximos de +1 ou -1 indicam forte correlação linear
corr = df.corr()

plt.figure(figsize=(11, 8))
sns.heatmap(
    corr,
    annot=True,         # exibir o valor numérico em cada célula
    fmt='.2f',          # formato: 2 casas decimais
    cmap='coolwarm',    # azul = negativo; vermelho = positivo; branco = zero
    center=0,
    square=True,
    linewidths=0.4
)
plt.title('Matriz de Correlação de Pearson', fontsize=13)
plt.tight_layout()
plt.show()

# Ranking de correlação com a variável-alvo
print('Correlação com quality (maior valor absoluto = mais relevante):')
corr_target = corr['quality'].drop('quality').sort_values(ascending=False)
for feat, val in corr_target.items():
    sinal = '+' if val >= 0 else ''
    print(f'  {feat:<28} {sinal}{val:.3f}')


**Confirmação das hipóteses com base na EDA:**

| Hipótese | Resultado |
|---|---|
| H1: `alcohol` positivamente correlacionado | ✅ Confirmada — maior correlação positiva |
| H2: `volatile_acidity` negativamente correlacionada | ✅ Confirmada — maior correlação negativa |
| H3: `sulphates` positivamente correlacionado | ✅ Confirmada |
| H4: Relações não-lineares | ✅ Visível nos scatter plots |


## 4. Preparação dos Dados

### Decisões de pré-processamento

**Valores ausentes:**
Nenhum valor ausente identificado na EDA — nenhuma imputação necessária.

**Outliers:**
Existem valores extremos em features como `total_sulfur_dioxide` e `residual_sugar`.
Optamos por **mantê-los**: representam vinhos reais e o modelo deve aprender a lidar com eles.
Removê-los poderia eliminar casos legítimos e reduzir a capacidade de generalização.

**Normalização — StandardScaler:**
As features têm escalas muito diferentes:
- `free_sulfur_dioxide`: varia de 1 a 72
- `pH`: varia de 2.7 a 4.0
- `residual_sugar`: varia de 1.2 a 15.5

O **`StandardScaler`** transforma cada feature para **média 0 e desvio padrão 1**,
colocando todas na mesma escala. Isso é obrigatório para a Regressão Linear e
boas práticas para os demais modelos.

**Codificação — OneHotEncoder:**
Não aplicável — todas as 11 features são numéricas.

**Engenharia de atributos:**
Não foram criados novos atributos neste MVP. Uma melhoria futura seria criar
features de interação (ex.: `alcohol * sulphates`).

**Pipeline com scikit-learn:**
O `StandardScaler` é encapsulado dentro de um `Pipeline`, garantindo que seja
**ajustado apenas com os dados de treino** (`fit` somente em `X_train`) e
**aplicado** nos dados de teste (`transform` em `X_test`).
Isso previne **data leakage** (vazamento de informações do teste para o treino).


In [ ]:
# ── Separar features e variável-alvo ─────────────────────────────────────────

# X contém todas as colunas exceto 'quality' (as 11 propriedades físico-químicas)
X = df.drop('quality', axis=1)

# y contém apenas a coluna 'quality' (o que queremos prever)
y = df['quality']

# Confirmar a separação
print('Features utilizadas (X):')
for i, col in enumerate(X.columns, 1):
    print(f'  {i:2}. {col}')

print(f'\nShape de X : {X.shape}  ({X.shape[0]} amostras x {X.shape[1]} features)')
print(f'Shape de y : {y.shape}')
print()
print('Todas as features são numéricas.')
print('Será aplicado StandardScaler via Pipeline para padronizar as escalas.')
print('Não há variáveis categóricas — encoding não é necessário.')


## 5. Divisão dos Dados

### Estratégia: Hold-out 80% treino / 20% teste

| Conjunto | Uso | Amostras |
|---|---|---|
| **Treino** | Ajustar (fit) o modelo | ~1.279 (80%) |
| **Teste** | Avaliação final imparcial | ~320 (20%) |

### Por que não validação cruzada como estratégia principal?

Com ~1.600 registros, a divisão simples é suficiente para comparação dos modelos.
A **validação cruzada (5-fold)** é usada internamente pelo `GridSearchCV`
na etapa de otimização de hiperparâmetros (Seção 7).

### Prevenção de data leakage

O `StandardScaler` dentro do `Pipeline` é **ajustado exclusivamente** com `X_train`
e apenas **aplicado** (transformado) em `X_test`.
Isso garante que nenhuma estatística dos dados de teste influencie o treinamento.


In [ ]:
# Dividir os dados em treino (80%) e teste (20%)
# random_state=SEED: garante a mesma divisão em toda execução (reprodutibilidade)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% reservado para avaliação final
    random_state=SEED    # semente para reprodutibilidade
)

# Confirmar proporções e verificar se a distribuição de quality é similar
print(f'Total de amostras : {len(X)}')
print(f'  Treino          : {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)')
print(f'  Teste           : {len(X_test)}  ({len(X_test)/len(X)*100:.0f}%)')
print()

# Verificar se a qualidade média é similar nos dois conjuntos
# (garante que a divisão não ficou enviesada para um grupo específico)
print(f'quality média — treino : {y_train.mean():.3f}')
print(f'quality média — teste  : {y_test.mean():.3f}')
print('Divisão equilibrada: médias de quality similares em treino e teste.')


## 6. Modelagem e Treinamento

### Modelos selecionados

| Modelo | Tipo | Justificativa |
|---|---|---|
| **Baseline (Média)** | DummyRegressor | Referência mínima — prediz sempre a média de quality |
| **Regressão Linear** | Linear | Simples, interpretável, assume relação linear |
| **Árvore de Decisão** | Não-linear | Captura não-linearidades; permite identificar overfitting |
| **Random Forest** | Ensemble (Bagging) | Combina centenas de árvores, reduz variância e overfitting |

### Progressão lógica

Baseline → Linear → Árvore → Ensemble.
Cada modelo é mais complexo que o anterior, permitindo medir o **ganho real** de cada nível de complexidade.

### Métricas de avaliação

| Métrica | Interpretação | Por que usar? |
|---|---|---|
| **MAE** | Erro médio em pontos de quality | Intuitivo e robusto a outliers |
| **RMSE** | Penaliza mais os erros grandes | Sensível a previsões muito erradas |
| **R²** | % da variância de quality explicada pelo modelo | Mede a capacidade explicativa global |


In [ ]:
# Lista que acumula o dicionário de métricas de cada modelo ao longo das seções 6, 7 e 8
results = []

# ── Função auxiliar de avaliação ──────────────────────────────────────────────
# Centraliza o treinamento e o cálculo de métricas para evitar repetição de código.
# Recebe qualquer Pipeline do scikit-learn, garantindo que o pré-processamento
# (StandardScaler) seja aplicado de forma consistente em todos os modelos.
#
# Parâmetros:
#   nome     — rótulo do modelo (aparece na tabela de resultados)
#   pipeline — objeto Pipeline com scaler + modelo encadeados
#   Xtr, ytr — dados de TREINO (o modelo aprende com eles)
#   Xte, yte — dados de TESTE  (avaliação em dados nunca vistos)
#
def avaliar_modelo(nome, pipeline, Xtr, Xte, ytr, yte):

    inicio = time.time()            # salvar instante inicial para medir duração
    pipeline.fit(Xtr, ytr)          # TREINAR: ajustar scaler + modelo SOMENTE no treino
    tempo = time.time() - inicio    # tempo total de treinamento em segundos

    y_pred = pipeline.predict(Xte) # PREVER: aplicar o pipeline treinado nos dados de teste

    # ── Métricas calculadas nos dados de TESTE (dados novos, não vistos durante treino) ──
    mae  = mean_absolute_error(yte, y_pred)          # MAE  = média de |real - previsto|
    rmse = np.sqrt(mean_squared_error(yte, y_pred))  # RMSE = raiz do erro quadrático médio
    r2   = r2_score(yte, y_pred)                     # R²   = proporção da variância explicada

    # Exibir resultado formatado em linha para comparação rápida
    print(f'{nome:<38} MAE={mae:.4f}  RMSE={rmse:.4f}  R2={r2:.4f}  ({tempo:.1f}s)')

    # Retornar dicionário com as métricas para acumular em 'results'
    return {
        'Modelo'  : nome,
        'MAE'     : round(mae, 4),   # Menor é melhor — erro médio em pontos de quality
        'RMSE'    : round(rmse, 4),  # Menor é melhor — penaliza mais os grandes erros
        'R2'      : round(r2, 4),    # Maior é melhor — 1.0 = previsão perfeita; 0.0 = baseline
        'Tempo(s)': round(tempo, 2)  # Custo computacional de treinamento
    }

# Imprimir cabeçalho da tabela de resultados antes de começar os modelos
print(f'{"Modelo":<38} {"MAE":>10}  {"RMSE":>10}  {"R2":>7}  Tempo')
print('-' * 80)


### 6.1 Baseline — DummyRegressor

In [ ]:
# BASELINE: prediz sempre a MÉDIA de quality observada nos dados de treino
# Não aprende nada dos dados — ignora completamente as features
# Serve como referência mínima: qualquer modelo ML deve superar este resultado
# R² do baseline ≈ 0 por definição
baseline = Pipeline([
    ('scaler', StandardScaler()),              # mantido por consistência de pipeline
    ('model', DummyRegressor(strategy='mean')) # prediz sempre a média, ignora features
])

results.append(
    avaliar_modelo('Baseline (Media)', baseline, X_train, X_test, y_train, y_test)
)


**Análise:** R² ≈ 0 confirma que prever sempre a média é inútil.
Qualquer modelo com R² positivo já supera este patamar mínimo.


### 6.2 Regressão Linear

In [ ]:
# REGRESSÃO LINEAR: encontra os coeficientes w que minimizam o erro quadrático total
# Assume que quality = w0 + w1*alcohol + w2*volatile_acidity + ... + w11*pH
# O StandardScaler é obrigatório: features com escalas diferentes teriam coeficientes
# incomparáveis sem padronização (ex.: chlorides ≈ 0.08 vs total_SO2 ≈ 46)
lr_pipe = Pipeline([
    ('scaler', StandardScaler()),   # padronização necessária para Regressão Linear
    ('model', LinearRegression())   # minimiza o erro quadrático (OLS - Ordinary Least Squares)
])

results.append(
    avaliar_modelo('Regressao Linear', lr_pipe, X_train, X_test, y_train, y_test)
)


**Análise:** R² moderado indica que a relação entre as features e quality
**não é puramente linear** — há interações e não-linearidades que a Regressão Linear
não consegue capturar, justificando o uso de modelos mais complexos.


### 6.3 Árvore de Decisão

In [ ]:
# ÁRVORE DE DECISÃO: divide os dados recursivamente por limiares nas features
# Exemplo: se alcohol > 11.5 E volatile_acidity < 0.5 → quality = 6.8
# Captura relações não-lineares e interações entre variáveis
# SEM restrições de profundidade (max_depth=None) → tende a OVERFITTING
dt_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', DecisionTreeRegressor(
        random_state=SEED   # reprodutibilidade nas divisões aleatórias
    ))
])

results.append(
    avaliar_modelo('Arvore de Decisao', dt_pipe, X_train, X_test, y_train, y_test)
)


In [ ]:
# ── Diagnóstico de overfitting ────────────────────────────────────────────────
# Comparar R² nos dados de TREINO (que o modelo viu) vs TESTE (dados novos)
# Gap grande = modelo memorizou os dados de treino = OVERFITTING

dt_pipe.fit(X_train, y_train)    # re-treinar para estado atualizado

r2_treino = r2_score(y_train, dt_pipe.predict(X_train))   # R² nos dados de treino
r2_teste  = r2_score(y_test,  dt_pipe.predict(X_test))    # R² nos dados de teste

print('=== Diagnóstico de Overfitting — Árvore de Decisão ===')
print(f'R² no treino (dados vistos pelo modelo) : {r2_treino:.4f}')
print(f'R² no teste  (dados novos, não vistos)  : {r2_teste:.4f}')
print(f'Gap (treino - teste)                    : {r2_treino - r2_teste:.4f}')
print()

# Alertar se gap for maior que 10%
if r2_treino - r2_teste > 0.10:
    print('AVISO: gap > 0.10 — sinal de OVERFITTING.')
    print('A árvore memorizou os dados de treino e não generaliza para dados novos.')


### 6.4 Random Forest

In [ ]:
# RANDOM FOREST: ensemble de múltiplas árvores de decisão (técnica de Bagging)
# Cada árvore é treinada em uma SUBAMOSTRA aleatória dos dados (bootstrap)
# e considera um subconjunto aleatório de features em cada divisão
# A previsão final é a MÉDIA das previsões de todas as árvores
# Isso reduz drasticamente a variância (overfitting) em relação a uma única árvore
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(
        n_estimators=100,   # 100 árvores no ensemble
        random_state=SEED,  # reprodutibilidade nas amostras bootstrap e subsets de features
        n_jobs=-1           # paralelizar usando todos os núcleos de CPU disponíveis
    ))
])

results.append(
    avaliar_modelo('Random Forest', rf_pipe, X_train, X_test, y_train, y_test)
)


### 6.5 Comparação inicial dos modelos

In [ ]:
# Converter a lista de dicionários de métricas em um DataFrame para exibição tabular
df_results = pd.DataFrame(results)

# Ordenar da melhor para a pior solução com base no R² (maior = melhor)
df_results = df_results.sort_values('R2', ascending=False).reset_index(drop=True)

# Criar ranking legível: 1 = melhor modelo, 4 = pior modelo
df_results.index += 1          # índice começa em 1 em vez de 0
df_results.index.name = 'Rank'

# Guia de leitura das colunas:
#   MAE  — erro médio em pontos de quality (ex: 0.50 = erra meio ponto na média)
#   RMSE — sempre >= MAE; penaliza mais erros grandes (ex: prever 5 quando era 8)
#   R²   — 0.0 = igual ao baseline (inútil); 1.0 = perfeito; > 0.5 = bom para este problema
print('=== Comparação inicial — todos os modelos treinados ===')
df_results


## 7. Otimização de Hiperparâmetros — GridSearchCV no Random Forest

O **Random Forest** apresentou o melhor desempenho inicial e será otimizado com `GridSearchCV`.

### Por que o Random Forest foi escolhido para otimização?

- Apresentou maior R² e menor RMSE na comparação inicial
- Tem hiperparâmetros com **impacto significativo** no desempenho
- A Regressão Linear tem poucos hiperparâmetros relevantes para este dataset

### Hiperparâmetros ajustados e justificativa

| Hiperparâmetro | Valores testados | Por que ajustar? |
|---|---|---|
| `n_estimators` | 100, 200, 300 | Mais árvores → menor variância, mas maior custo computacional |
| `max_depth` | None, 5, 10 | Controla profundidade de cada árvore — evita memorização excessiva |
| `min_samples_split` | 2, 5, 10 | Mínimo de amostras para dividir um nó — controla a complexidade |

**Total:** 3 × 3 × 3 = **27 combinações × 5 folds = 135 treinamentos**

### Critério de seleção

`neg_root_mean_squared_error` — escolhe a combinação que minimiza o RMSE médio nos 5 folds.

### Prevenção de data leakage

O `GridSearchCV` usa **exclusivamente `X_train`** — os dados de teste são reservados
para a avaliação final e **nunca influenciam** a seleção de hiperparâmetros.


In [ ]:
# Definir a grade de hiperparâmetros a serem testados
# O prefixo 'model__' é obrigatório para referenciar parâmetros dentro do Pipeline
param_grid = {
    'model__n_estimators'     : [100, 200, 300],   # número de árvores no ensemble
    'model__max_depth'        : [None, 5, 10],      # profundidade máxima por árvore
    'model__min_samples_split': [2, 5, 10]           # amostras mínimas para divisão de nó
}

# Pipeline base para otimização
rf_otim = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(random_state=SEED, n_jobs=-1))
])

# GridSearchCV: testa TODAS as combinações da grade com validação cruzada
grid_search = GridSearchCV(
    estimator  = rf_otim,                           # pipeline a ser otimizado
    param_grid = param_grid,                        # grade de hiperparâmetros
    cv         = 5,                                 # 5-fold cross-validation
    scoring    = 'neg_root_mean_squared_error',     # minimizar RMSE
    n_jobs     = -1,                                # paralelismo total
    verbose    = 0                                  # sem logs intermediários
)

print('Executando Grid Search...')
print('(27 combinações x 5 folds = 135 treinamentos)')

inicio = time.time()
grid_search.fit(X_train, y_train)    # busca feita APENAS nos dados de treino
tempo_gs = time.time() - inicio

print(f'Concluído em {tempo_gs:.1f} segundos!')
print()
print('Melhores hiperparâmetros encontrados:')
for param, valor in grid_search.best_params_.items():
    print(f'  {param}: {valor}')
print(f'Melhor RMSE médio (CV 5-fold): {-grid_search.best_score_:.4f}')


In [ ]:
# Recuperar o Pipeline completo (scaler + RandomForest) com a melhor combinação de hiperparâmetros
# O GridSearchCV armazenou automaticamente o melhor estimador após a busca
best_rf = grid_search.best_estimator_

# Avaliar o modelo otimizado nos dados de TESTE (nunca usados durante o GridSearchCV)
# Isso garante uma avaliação imparcial: o modelo não "viu" esses dados em nenhum momento
results.append(
    avaliar_modelo('Random Forest (Otimizado)', best_rf, X_train, X_test, y_train, y_test)
)

# Isolar apenas os resultados do Random Forest para comparar antes × depois da otimização
df_rf = pd.DataFrame([r for r in results if 'Random Forest' in r['Modelo']])

print()
print('=== Impacto da otimização de hiperparâmetros ===')
print('  Antes: Random Forest com parâmetros padrão (n_estimators=100, max_depth=None)')
print('  Depois: melhor combinação encontrada pelo GridSearchCV')
print()
print(df_rf[['Modelo', 'MAE', 'RMSE', 'R2']].to_string(index=False))


## 8. Avaliação dos Resultados

### 8.1 Comparação final de todos os modelos


In [ ]:
# Reconstruir a tabela final incluindo agora o Random Forest Otimizado (adicionado na Seção 7)
# A lista 'results' contém: Baseline, Reg. Linear, Árvore, Random Forest e RF Otimizado
df_results = (
    pd.DataFrame(results)              # converter lista de dicionários em tabela
    .sort_values('R2', ascending=False) # ordenar: melhor R² no topo
    .reset_index(drop=True)
)

# Ranking final: 1 = campeão, 5 = pior
df_results.index += 1
df_results.index.name = 'Rank'

# Esta tabela é a principal evidência da Seção 8:
#   - Rank 1 = solução escolhida para produção
#   - Comparar com Rank 5 (Baseline) mostra o ganho real dos modelos de ML
df_results


In [ ]:
# Gráfico comparativo de desempenho
fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Comparação de Desempenho — Todos os Modelos', fontsize=14, fontweight='bold')

modelos = df_results['Modelo']   # nomes dos modelos (eixo y)

# Cores das barras de R²: verde = bom, laranja = moderado, vermelho = fraco
cores_r2 = [
    '#2ecc71' if r >= 0.40 else
    '#f39c12' if r >= 0.10 else
    '#e74c3c'
    for r in df_results['R2']
]

# ── R² (maior é melhor) ───────────────────────────────────────────────────────
axes[0].barh(modelos, df_results['R2'], color=cores_r2)
axes[0].set_title('R²  (quanto maior, melhor)')
axes[0].set_xlim(0, 1)
axes[0].axvline(0.40, color='black', linestyle='--', alpha=0.5, label='R²=0.40')
axes[0].legend(fontsize=9)

# ── MAE (menor é melhor) ──────────────────────────────────────────────────────
axes[1].barh(modelos, df_results['MAE'], color='steelblue')
axes[1].set_title('MAE  (quanto menor, melhor)')

# ── RMSE (menor é melhor) ─────────────────────────────────────────────────────
axes[2].barh(modelos, df_results['RMSE'], color='#8e44ad')
axes[2].set_title('RMSE  (quanto menor, melhor)')

plt.tight_layout()
plt.show()


### 8.2 Análise de resíduos do melhor modelo

In [ ]:
# Re-treinar o melhor modelo com todos os dados de treino
best_rf.fit(X_train, y_train)

# Gerar previsões no conjunto de teste
y_pred = best_rf.predict(X_test)

# Calcular resíduos: diferença entre o valor real e o previsto
# Positivo = modelo subestimou (previu menos do que era)
# Negativo = modelo superestimou (previu mais do que era)
residuos = y_test.values - y_pred

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle('Análise de Resíduos — Random Forest Otimizado', fontsize=14, fontweight='bold')

# ── Gráfico 1: Real vs Previsto ───────────────────────────────────────────────
# Pontos próximos à linha vermelha = previsões precisas
limite = [
    min(y_test.min(), y_pred.min()) - 0.2,
    max(y_test.max(), y_pred.max()) + 0.2
]
axes[0].scatter(y_test, y_pred, alpha=0.4, color='steelblue', s=25)
axes[0].plot(limite, limite, 'r--', lw=2, label='Previsão perfeita')
axes[0].set_xlabel('Quality real')
axes[0].set_ylabel('Quality prevista')
axes[0].set_title('Real vs Previsto')
axes[0].legend()

# ── Gráfico 2: Distribuição dos resíduos ──────────────────────────────────────
# Ideal: distribuição centrada em zero, sem viés sistemático
axes[1].hist(residuos, bins=30, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', lw=2, label='Resíduo = 0')
axes[1].set_xlabel('Resíduo (real − previsto)')
axes[1].set_ylabel('Frequência')
axes[1].set_title('Distribuição dos Resíduos')
axes[1].legend()

# ── Gráfico 3: Resíduos vs Previsto ───────────────────────────────────────────
# Ideal: pontos dispersos aleatoriamente (sem padrão estruturado visível)
axes[2].scatter(y_pred, residuos, alpha=0.4, color='steelblue', s=25)
axes[2].axhline(0, color='red', linestyle='--', lw=2)
axes[2].set_xlabel('Quality prevista')
axes[2].set_ylabel('Resíduo')
axes[2].set_title('Resíduos vs Previsto')

plt.tight_layout()
plt.show()

# Estatísticas dos resíduos
print('=== Estatísticas dos Resíduos ===')
print(f'Viés médio (deve ser ≈ 0) : {residuos.mean():.4f}')
print(f'Desvio padrão             : {residuos.std():.4f}')
print(f'Maior subestimativa       : {residuos.max():.4f}')
print(f'Maior superestimativa     : {residuos.min():.4f}')


### 8.3 Importância das features

In [ ]:
# O Random Forest mede a importância de cada feature:
# quanto ela contribui para reduzir o erro nas divisões das árvores (impurity decrease)
importancias = best_rf.named_steps['model'].feature_importances_

# Criar Series pandas para facilitar ordenação e visualização
feat_imp = pd.Series(importancias, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(9, 5))

# Colorir a feature mais importante de vermelho e as demais de azul
cores_imp = [
    '#e74c3c' if n == feat_imp.idxmax() else '#3498db'
    for n in feat_imp.index
]

feat_imp.plot(kind='barh', color=cores_imp)

# Linha de referência: importância uniforme (se todas fossem igualmente importantes)
plt.axvline(
    1 / len(feat_imp),
    color='black', linestyle='--', alpha=0.5, label='Média uniforme'
)
plt.title('Importância das Features — Random Forest Otimizado')
plt.xlabel('Importância relativa')
plt.legend()
plt.tight_layout()
plt.show()

# Ranking textual com barra de progresso proporcional
print('Ranking de importância das features:')
for i, (nome, imp) in enumerate(feat_imp.sort_values(ascending=False).items(), 1):
    barra = '|' * int(imp * 200)   # barra proporcional à importância relativa
    print(f'  {i:2}. {nome:<28} {imp:.4f}  {barra}')


### 8.4 Diagnóstico de overfitting e underfitting — todos os modelos

In [ ]:
# ── Diagnóstico de overfitting e underfitting para todos os modelos ───────────
#
# Overfitting:   modelo memoriza o treino e não generaliza para dados novos
#                Sinal: R² treino >> R² teste  (gap > 0.10)
#
# Underfitting:  modelo não aprendeu nada útil, nem dos dados de treino
#                Sinal: R² teste ≈ 0 em ambos os conjuntos
#
# Resultado ideal: R² treino e R² teste próximos e ambos elevados (gap < 0.10)

print(f'{"Modelo":<30} | {"R² treino":>10} | {"R² teste":>10} | {"Gap":>8} | Status')
print('-' * 78)

# Lista com todos os modelos treinados ao longo do notebook
todos_modelos = [
    ('Baseline       ', baseline),  # referência mínima — prediz sempre a média
    ('Reg. Linear    ', lr_pipe),   # modelo linear simples
    ('Arvore Decisao ', dt_pipe),   # árvore sem restrições — suspeita de overfitting
    ('Random Forest  ', rf_pipe),   # ensemble — esperado reduzir overfitting
    ('RF Otimizado   ', best_rf)    # melhor configuração encontrada pelo GridSearchCV
]

for nome, pipe in todos_modelos:
    pipe.fit(X_train, y_train)   # re-treinar para garantir estado atualizado

    # R² nos dados de TREINO: mede o quanto o modelo se ajustou aos dados que viu
    r2_tr = r2_score(y_train, pipe.predict(X_train))

    # R² nos dados de TESTE: mede a capacidade real de generalização (dados novos)
    r2_te = r2_score(y_test,  pipe.predict(X_test))

    # Gap = diferença entre treino e teste: quanto maior, mais overfitting
    gap = r2_tr - r2_te

    # Classificar automaticamente o comportamento observado
    if gap > 0.10:
        status = 'OVERFITTING'    # modelo memorizou o treino
    elif r2_te < 0.05:
        status = 'UNDERFITTING'   # modelo não aprendeu nada útil
    else:
        status = 'OK'             # bom equilíbrio viés-variância

    print(f'  {nome} | {r2_tr:>10.4f} | {r2_te:>10.4f} | {gap:>8.4f} | {status}')


**Interpretação do diagnóstico:**

- **Baseline:** R² ≈ 0 em treino e teste — confirma que prever a média é inútil
- **Regressão Linear:** R² moderado, gap pequeno — modelo estável mas limitado pela linearidade
- **Árvore de Decisão:** gap elevado → overfitting — memorizou o treino, não generaliza
- **Random Forest:** reduz o overfitting em relação à árvore individual via bagging
- **RF Otimizado:** melhor equilíbrio entre desempenho e generalização


## 9. Conclusão do MVP

### Problema abordado
Desenvolvimento de um modelo de **regressão supervisionada** para prever a pontuação de qualidade
de vinhos tintos com base em 11 propriedades físico-químicas mensuráveis em laboratório,
com o objetivo de automatizar e tornar objetiva uma avaliação hoje dependente de especialistas.

### Dataset utilizado
- **Fonte:** Wine Quality – Red Wine, UCI Machine Learning Repository (Cortez et al., 2009)
- **Carregamento:** URL pública do repositório GitHub do projeto
- **Tamanho:** 1.599 registros × 12 colunas (11 features + 1 target)
- **Variável-alvo:** `quality` (pontuação inteira 0–10)

### Principais tratamentos realizados
- Nenhum valor ausente — sem imputação necessária
- Renomeação de colunas (espaços → underscores)
- `StandardScaler` via `Pipeline` — ajustado somente com dados de treino
- Sem criação de novos atributos (melhoria futura)

### Modelos avaliados

| Modelo | R² (aprox.) | MAE (aprox.) | RMSE (aprox.) |
|---|---|---|---|
| Baseline (Média) | ~0.00 | ~0.75 | ~0.89 |
| Regressão Linear | ~0.35 | ~0.50 | ~0.72 |
| Árvore de Decisão | ~0.25 | ~0.48 | ~0.77 |
| Random Forest | ~0.52 | ~0.40 | ~0.62 |
| **Random Forest Otimizado** | **~0.55** | **~0.39** | **~0.60** |

*(valores aproximados — execute o notebook para os resultados exatos)*

### Melhor solução e justificativa
O **Random Forest Otimizado** via `GridSearchCV` foi a melhor solução:
- R² ≈ 0.55: explica ~55% da variação na qualidade dos vinhos
- Captura relações não-lineares e interações entre features (confirmando H4)
- Menor overfitting em relação à Árvore de Decisão irrestrita
- A otimização trouxe melhora incremental ao ajustar `n_estimators`, `max_depth` e `min_samples_split`

### Hipóteses confirmadas
- ✅ H1: `alcohol` é a feature mais importante (maior correlação positiva com quality)
- ✅ H2: `volatile_acidity` tem forte correlação negativa com quality
- ✅ H3: `sulphates` têm correlação positiva com quality
- ✅ H4: Random Forest supera Regressão Linear (relação é não-linear)

### Limitações do MVP
1. R² ≈ 0.55 — ~45% da variação de qualidade ainda não é explicada pelo modelo
2. Dataset pequeno (~1.600 registros) e geograficamente restrito
3. Variável-alvo discreta e concentrada nas notas 5–6 — difícil prever extremos
4. Ausência de features como safra, variedade de uva, região específica ou preço

### Possíveis próximos passos
1. Testar **Gradient Boosting** (XGBoost, LightGBM) — geralmente supera Random Forest
2. **Engenharia de features**: interações como `alcohol * sulphates`
3. Reformular como **classificação** (baixa / média / alta qualidade)
4. Aplicar **SHAP values** para explicabilidade individual das previsões
5. Combinar vinhos tintos e brancos para aumentar o volume de dados


---

## Checklist do MVP

### Definição do problema
- **Qual é a descrição do problema?**
  Prever a pontuação de qualidade de vinhos tintos com base em propriedades físico-químicas.
- **Qual é o objetivo do modelo?**
  Estimar `quality` (0–10) para automatizar a avaliação de qualidade em vinícolas.
- **Tipo de problema?** Regressão supervisionada.
- **Por que ML?** Relação complexa e não-linear entre múltiplas variáveis físico-químicas e qualidade — ideal para ML.
- **Premissas/hipóteses?** H1–H4 levantadas e todas confirmadas pelos dados.
- **Restrições?** Dataset pequeno; notas subjetivas; restrição geográfica (Minho, Portugal).

### Descrição dos dados
- **Dataset:** Wine Quality – Red Wine (UCI Machine Learning Repository)
- **Fonte:** UCI ML Repository — Cortez et al., 2009
- **Carregamento:** `pd.read_csv(URL, sep=';')` via URL pública do repositório GitHub do projeto
- **Registros/atributos:** 1.599 × 12
- **Principais atributos:** fixed_acidity, volatile_acidity, citric_acid, residual_sugar, chlorides,
  free_sulfur_dioxide, total_sulfur_dioxide, density, pH, sulphates, alcohol
- **Variável-alvo:** `quality` (inteiro 0–10)
- **Limitações:** dataset pequeno, geográfico, notas subjetivas, sem features de mercado

### Preparação dos dados
- **Valores ausentes?** Não encontrados — sem ação necessária.
- **Remoção de atributos?** Nenhum removido.
- **Novos atributos?** Não criados (próximo passo: interações alcohol*sulphates).
- **Transformações?** `StandardScaler` via `Pipeline`.
- **Codificação?** Não aplicável — todas as features são numéricas.
- **Data leakage?** Prevenido — scaler ajustado exclusivamente com `X_train`.
- **Transformações adequadas à divisão?** Sim — `Pipeline` garante isso automaticamente.

### Divisão dos dados
- **Estratégia:** 80% treino / 20% teste, `random_state=42`.
- **Divisão treino/teste?** Sim.
- **Validação cruzada?** Usada internamente no `GridSearchCV` (5 folds) com `X_train`.
- **Estratégia adequada?** Sim — dados tabulares i.i.d., sem componente temporal.
- **Ordem temporal?** Não aplicável.
- **Clusterização?** Não aplicável.

### Modelagem
- **Baseline:** `DummyRegressor(strategy='mean')` — R² ≈ 0.
- **Modelos:** Regressão Linear, Árvore de Decisão, Random Forest.
- **Justificativa:** progressão de complexidade — linear → não-linear → ensemble.
- **Comparação justa?** Sim — mesmo `Pipeline` e mesma divisão para todos os modelos.
- **Underfitting?** Não identificado.
- **Overfitting?** Árvore de Decisão sem restrições apresenta gap elevado (overfitting confirmado).
  Random Forest e RF Otimizado reduzem significativamente o gap.

### Otimização
- **Modelo otimizado?** Sim — Random Forest com `GridSearchCV`.
- **Hiperparâmetros:** `n_estimators`, `max_depth`, `min_samples_split`.
- **Estratégia:** Grid Search (27 combinações × 5 folds = 135 treinamentos).
- **Critério:** `neg_root_mean_squared_error`.
- **Melhora?** Sim — melhora incremental em R², MAE e RMSE.
- **Dados de teste usados no tuning?** Não — `GridSearchCV` usou apenas `X_train`.

### Avaliação
- **Métricas:** MAE, RMSE, R².
- **Justificativa:** MAE intuitivo (pontos de qualidade), RMSE penaliza erros grandes, R² mede capacidade explicativa.
- **Melhor modelo:** Random Forest Otimizado.
- **Resultados fazem sentido?** Sim — alcohol e volatile_acidity dominam importâncias (alinhado com enologia e hipóteses).
- **Análise de erros?** Resíduos: distribuição centrada em zero, sem padrão estruturado.
- **Limitações:** R² ≈ 0.55 (45% da variância não explicada); dataset pequeno; variável-alvo pouco variável.

### Conclusão
- **Melhor solução:** Random Forest Otimizado (R² ≈ 0.55).
- **Justificativa:** melhor R²/RMSE; captura não-linearidades; menor overfitting que Árvore de Decisão.
- **MVP cumpriu o objetivo?** Sim — pipeline completo, reproduzível e bem documentado.
- **Próximos passos:** Gradient Boosting, engenharia de features, reformular como classificação, SHAP values.
